In [4]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

DATA_ROOT = Path("/home/hice1/salammurillo3/scratch/bendr/physionet")
SUBJECTS = [s for s in range(1, 110) if s not in (88, 90, 92, 100)]
RUNS = [6, 10, 14]
BANDS = {"mu": (8, 13), "beta": (13, 30)}


def load_subject(subject):
    subject_dir = DATA_ROOT / f"S{subject:03d}"
    paths = [
        subject_dir / f"S{subject:03d}R{run:02d}.edf"
        for run in RUNS
    ]

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing EDF files: {missing}")

    raws = [
        mne.io.read_raw_edf(path, preload=True)
        for path in paths
    ]

    raw = mne.concatenate_raws(raws)
    mne.datasets.eegbci.standardize(raw)
    raw.set_montage("standard_1005", on_missing="ignore")

    events, event_id = mne.events_from_annotations(raw)
    wanted = {
        label: code
        for label, code in event_id.items()
        if label in ("T1", "T2")
    }

    epochs = mne.Epochs(
        raw,
        events,
        wanted,
        tmin=0.0,
        tmax=6.0 - 1.0 / raw.info["sfreq"],
        baseline=None,
        preload=True,
    )

    psd = epochs.compute_psd(method="welch", fmin=7, fmax=30)
    data, freqs = psd.get_data(return_freqs=True)

    feats = [
        np.log(data[:, :, (freqs >= lo) & (freqs <= hi)].mean(axis=2))
        for lo, hi in BANDS.values()
    ]
    X = np.concatenate(feats, axis=1)

    inv = {code: label for label, code in wanted.items()}
    y = np.array([
        0 if inv[event_code] == "T1" else 1
        for event_code in epochs.events[:, 2]
    ])

    return X, y


def main():
    X_all = []
    y_all = []
    groups = []

    for subject in SUBJECTS:
        try:
            X, y = load_subject(subject)
            X_all.append(X)
            y_all.append(y)
            groups.append(np.full(len(y), subject))
            print(f"Subject {subject:03d}: {len(y)} epochs")
        except Exception as error:
            print(f"Skipping subject {subject:03d}: {error}")

    if not X_all:
        raise RuntimeError("No subject data could be loaded.")

    X = np.vstack(X_all)
    y = np.concatenate(y_all)
    groups = np.concatenate(groups)

    print(f"\nFeature matrix: {X.shape}")
    print(f"Class counts: T1={np.sum(y == 0)}, T2={np.sum(y == 1)}")

    subject_folds = [
        fold.tolist()
        for fold in np.array_split(np.array(SUBJECTS), 5)
    ]
    fold_rows = []

    for fold, test_subjects in enumerate(subject_folds, start=1):
        validation_subjects = subject_folds[fold - 2]
        train_subjects = [
            subject
            for subject in SUBJECTS
            if subject not in test_subjects
            and subject not in validation_subjects
        ]

        train_mask = np.isin(groups, train_subjects)
        test_mask = np.isin(groups, test_subjects)

        rf = RandomForestClassifier(
            n_estimators=300,
            random_state=0,
            n_jobs=-1,
        )
        rf.fit(X[train_mask], y[train_mask])
        predictions = rf.predict(X[test_mask])

        accuracy = accuracy_score(y[test_mask], predictions)
        balanced_accuracy = balanced_accuracy_score(y[test_mask], predictions)

        fold_rows.append({
            "fold": fold,
            "train_subjects": len(train_subjects),
            "validation_subjects": len(validation_subjects),
            "test_subjects": len(test_subjects),
            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
        })

        print(
            f"Fold {fold}: accuracy={accuracy:.3f}, "
            f"balanced_accuracy={balanced_accuracy:.3f}, "
            f"test_subjects={test_subjects}"
        )

    results = pd.DataFrame(fold_rows)

    print("\nRandom Forest cross-subject results")
    print(results.to_string(index=False))
    print(
        f"\nMean accuracy: {results['accuracy'].mean():.3f} "
        f"+/- {results['accuracy'].std():.3f}"
    )
    print(
        f"Mean balanced accuracy: "
        f"{results['balanced_accuracy'].mean():.3f} "
        f"+/- {results['balanced_accuracy'].std():.3f}"
    )

    results.to_csv("random_forest_results.csv", index=False)
    print("\nSaved: random_forest_results.csv")

    return results


main()

Extracting EDF parameters from /home/hice1/salammurillo3/scratch/bendr/physionet/S001/S001R06.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /home/hice1/salammurillo3/scratch/bendr/physionet/S001/S001R10.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /home/hice1/salammurillo3/scratch/bendr/physionet/S001/S001R14.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: ['T0', 'T1', 'T2']
Not setting metadata
45 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 45 events and 960 original time points ...
3 bad epochs dropped
Effective window size : 6.

,fold,train_subjects,validation_subjects,test_subjects,accuracy,balanced_accuracy
0,1,63,21,21,0.540816,0.540816
1,2,63,21,21,0.583900,0.583900
2,3,63,21,21,0.561224,0.561224
3,4,63,21,21,0.530612,0.530612
4,5,63,21,21,0.541950,0.541950
